In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("survey_results.csv")

method_map = {
    "tts.wav":           "GATAS",
    "waveform.wav":      "Waveform",
    "smack.wav":         "SMACK",
    "pgd.wav":           "PGD",
    "harvard_audio.wav": "Ground Truth",
}
df["Method"] = df["File"].map(method_map)

grouped = df.groupby("Method")[["MOS_Technical", "MOS_Semantic"]].agg(["mean", "std"])
grouped.columns = ["tech_mean", "tech_std", "sem_mean", "sem_std"]
grouped = grouped.reindex(["Ground Truth", "GATAS", "Waveform", "SMACK", "PGD"])

def fmt(mean, std):
    return f"${mean:.2f} \\pm {std:.2f}$"

rows = []
for method, row in grouped.iterrows():
    rows.append(f"    {method} & {fmt(row.tech_mean, row.tech_std)} & {fmt(row.sem_mean, row.sem_std)} \\\\")

latex = "\n".join([
    "\\begin{table}[h]",
    "\\centering",
    "\\begin{tabular}{lcc}",
    "\\toprule",
    "    Method & MOS Technical & MOS Semantic \\\\",
    "\\midrule",
    *rows,
    "\\bottomrule",
    "\\end{tabular}",
    "\\caption{Mean Opinion Scores (MOS) by method.}",
    "\\end{table}",
])
print(latex)

## Automatic validity check

A generated failure is valid if a strong reference ASR that is not attacked (Whisper large-v3) still recognizes the
ground truth in the adversarial audio (WER $\leq$ 0.3, `src/metrics.py`). This complements the human study.

In [ ]:
import sys
sys.path.insert(0, ".")
from results_loader import load_results

for ds in ["harvard", "librispeech"]:
    records = load_results("../outputs/results", dataset=ds, asr_model="whisper-tiny")
    print(f"=== {ds} ===")
    for m, recs in records.items():
        if not recs:
            continue
        valid = [r["valid"] for r in recs]
        successful = [r for r in recs if r["success"]]
        print(f"{m:10s} valid {100 * np.mean(valid):5.1f}% | "
              f"valid among successful {100 * np.mean([r['valid'] for r in successful]) if successful else float('nan'):5.1f}% | "
              f"reference WER {np.mean([r['reference_wer'] for r in recs]):.3f} "
              f"(GT audio: {np.mean([r['validity']['reference_gt_wer'] for r in recs]):.3f})")